# Dataclass — Nível Intermediário

**Objetivo:** usar os parâmetros do decorador com intenção (imutabilidade, hash, memória, construtores seguros),
modelar herança, introspectar campos e aplicar dataclasses em **configuração** e **troca de dados (JSON)**.

Roteiro:

1. `frozen=True` e as regras de `__hash__`
2. `frozen` é raso: imutabilidade de verdade
3. `slots=True`: memória e atributos fixos
4. `kw_only` e o sentinela `KW_ONLY`
5. `InitVar`: parâmetros que não viram campo
6. Herança e o problema dos defaults
7. `ClassVar`: atributos de classe
8. `fields()` e `metadata`
9. `match/case` com `__match_args__`
10. **Configuração**: TOML → dataclasses aninhadas (e onde dói)
11. **JSON**: serializar e desserializar

In [1]:
import sys
from dataclasses import dataclass, field, fields, asdict, replace, FrozenInstanceError, KW_ONLY, InitVar
from typing import ClassVar

## 1. `frozen=True` e as regras de `__hash__`

Um objeto `frozen` não aceita atribuição depois de criado. Como ele não muda, o Python pode gerar um `__hash__`
seguro — e aí a instância pode ir em `set` e ser chave de `dict`.

In [2]:
@dataclass
class MutablePoint:
    x: float
    y: float

@dataclass(frozen=True)
class Point:
    x: float
    y: float

print(f"MutablePoint.__hash__: {MutablePoint.__hash__}")
try:
    {MutablePoint(0, 0)}
except TypeError as e:
    print(f"TypeError: {e}")

p = Point(1, 2)
try:
    p.x = 10
except FrozenInstanceError as e:
    print(f"FrozenInstanceError: {e}")

visited = {Point(0, 0), Point(1, 2), Point(1, 2)}
print(f"set de pontos: {visited}")
print(f"cópia alterada: {replace(p, x=10)}")
print(f"original: {p}")

MutablePoint.__hash__: None
TypeError: unhashable type: 'MutablePoint'
FrozenInstanceError: cannot assign to field 'x'
set de pontos: {Point(x=1, y=2), Point(x=0, y=0)}
cópia alterada: Point(x=10, y=2)
original: Point(x=1, y=2)


| `eq` | `frozen` | `unsafe_hash` | `__hash__` |
|---|---|---|---|
| True | False | False | `None` (não-hashable) |
| True | True | False | gerado dos campos ✅ |
| False | – | False | herdado de `object` (identidade) |
| – | – | True | gerado mesmo mutável ⚠️ |

Por que `unsafe_hash` é perigoso: se o objeto muda **depois** de entrar num `set`, o hash muda e o `set` não o encontra mais.

In [3]:
@dataclass(unsafe_hash=True)
class Danger:
    v: int

d = Danger(1)
s = {d}
d.v = 2
print(f"d in s: {d in s}  <- o objeto está no set, mas não é encontrado")

d in s: False  <- o objeto está no set, mas não é encontrado


**Correção:** se o objeto precisa ir em `set`/dict, ele deve ser **imutável** — `frozen=True` em vez de `unsafe_hash=True`.
Para "mudar", crie uma nova instância com `replace` (o set continua consistente).

In [4]:
@dataclass(frozen=True)
class Safe:
    v: int

d = Safe(1)
s = {d}
try:
    d.v = 2
except FrozenInstanceError as e:
    print(f"FrozenInstanceError: {e}")
d2 = replace(d, v=2)
print(f"d in s: {d in s}")
print(f"d2 in s: {d2 in s}")
s = (s - {d}) | {d2}            # trocar explicitamente
print(f"s: {s}")

FrozenInstanceError: cannot assign to field 'v'
d in s: True
d2 in s: False
s: {Safe(v=2)}


### Atribuindo em `__post_init__` de uma classe frozen
`self.campo = ...` é bloqueado até dentro da própria classe. A saída oficial é `object.__setattr__`:

In [5]:
@dataclass(frozen=True)
class Car:
    name: str
    color: str
    description: str = field(init=False)

    def __post_init__(self):
        object.__setattr__(self, "description", f"{self.name} - {self.color}")

print(f'Car("Saveiro", "amarela"): {Car("Saveiro", "amarela")}')

Car("Saveiro", "amarela"): Car(name='Saveiro', color='amarela', description='Saveiro - amarela')


## 2. `frozen` é raso

`frozen` impede **reatribuir** o campo, mas não impede **mutar** o objeto guardado nele:

In [6]:
@dataclass(frozen=True)
class Team:
    name: str
    members: list[str]

t = Team("core", ["ana"])
t.members.append("intruso")
print(f"t: {t}  <- 'imutável', mas mudou")

@dataclass(frozen=True)
class TeamSafe:
    name: str
    members: tuple[str, ...]

    def __post_init__(self):
        object.__setattr__(self, "members", tuple(self.members))   # aceita lista, guarda tupla

ts = TeamSafe("core", ["ana", "bia"])
print(f"ts: {ts}")
print(f"hash: {hash(ts)}")

t: Team(name='core', members=['ana', 'intruso'])  <- 'imutável', mas mudou
ts: TeamSafe(name='core', members=('ana', 'bia'))
hash: -2028989975406528216


## 3. `slots=True` (Python 3.10+)

Por padrão cada instância carrega um `__dict__` (um dicionário!) para guardar os atributos.
Com `slots=True`, a dataclass é recriada com `__slots__`: atributos ficam em posições fixas.

Efeitos:
* **menos memória** por instância
* acesso a atributo um pouco **mais rápido**
* **não** dá para criar atributos novos dinamicamente (pega typos!)

In [7]:
@dataclass
class WithDict:
    x: int
    y: int
    z: int

@dataclass(slots=True)
class WithSlots:
    x: int
    y: int
    z: int

a, b = WithDict(1, 2, 3), WithSlots(1, 2, 3)
print(f"WithDict: {sys.getsizeof(a) + sys.getsizeof(a.__dict__)} bytes (objeto + __dict__)")
print(f"WithSlots: {sys.getsizeof(b)} bytes")
print(f"__slots__: {WithSlots.__slots__}")

a.zz = 99                     # typo silencioso
try:
    b.zz = 99
except AttributeError as e:
    print(f"AttributeError: {e}")

WithDict: 344 bytes (objeto + __dict__)
WithSlots: 56 bytes
__slots__: ('x', 'y', 'z')
AttributeError: 'WithSlots' object has no attribute 'zz' and no __dict__ for setting new attributes


Os números de memória em escala (milhões de objetos) estão no notebook avançado.

> Restrição: com `slots=True` não existe `__dict__`, então `functools.cached_property` não funciona (veja o avançado).

## 4. `kw_only` e o sentinela `KW_ONLY` (3.10+)

Construtores com muitos parâmetros posicionais são uma fonte de bugs (`Config(5432, 30, True, False)` — qual é qual?).
`kw_only=True` obriga a passar por nome.

In [8]:
@dataclass(kw_only=True)
class RetryPolicy:
    max_attempts: int = 3
    backoff_seconds: float = 0.5
    retry_on_timeout: bool = True

print(f"RetryPolicy(max_attempts=5): {RetryPolicy(max_attempts=5)}")
try:
    RetryPolicy(5, 0.1, False)
except TypeError as e:
    print(f"TypeError: {e}")

RetryPolicy(max_attempts=5): RetryPolicy(max_attempts=5, backoff_seconds=0.5, retry_on_timeout=True)
TypeError: RetryPolicy.__init__() takes 1 positional argument but 4 were given


Para misturar (alguns posicionais, o resto nomeado), use o sentinela `_: KW_ONLY`:

In [9]:
@dataclass
class HttpRequest:
    method: str
    url: str
    _: KW_ONLY
    timeout: float = 10.0
    headers: dict[str, str] = field(default_factory=dict)
    verify_tls: bool = True

print(f'HttpRequest("GET", "https://api.example.com", timeout=2): {HttpRequest("GET", "https://api.example.com", timeout=2)}')

HttpRequest("GET", "https://api.example.com", timeout=2): HttpRequest(method='GET', url='https://api.example.com', timeout=2, headers={}, verify_tls=True)


## 5. `InitVar`: parâmetros que não viram campo

Às vezes o construtor precisa de um dado **só para calcular** outro campo (uma senha para gerar hash, uma conexão para buscar algo).
`InitVar` aparece no `__init__`, é repassado ao `__post_init__`, mas **não** é guardado nem aparece no `repr`/`asdict`.

In [10]:
import hashlib

@dataclass
class User:
    username: str
    password: InitVar[str]
    password_hash: str = field(init=False, repr=False)

    def __post_init__(self, password: str):
        if len(password) < 8:
            raise ValueError("senha curta")
        self.password_hash = hashlib.sha256(password.encode()).hexdigest()

u = User("dante", "s3nh4-longa")
print(f"u: {u}")
print(f"hash: {u.password_hash[:12]} ...")
print(f"asdict: {asdict(u)}")
print(f'tem atributo password?: {hasattr(u, "password")}')

u: User(username='dante')
hash: 549cab0e3c20 ...
asdict: {'username': 'dante', 'password_hash': '549cab0e3c20c7fa6feafe17f84d182dab5aae1a8edd9f114540700322eef3fd'}
tem atributo password?: False


## 6. Herança e o problema dos defaults

Os campos da classe filha são **adicionados ao final** dos campos da classe pai. Se o pai tem default e a filha não,
o `__init__` gerado ficaria inválido (`def __init__(self, a=0, b)`):

In [11]:
@dataclass
class BaseEntity:
    id: int | None = None

try:
    @dataclass
    class Product(BaseEntity):
        name: str
except TypeError as e:
    print(f"TypeError: {e}")

TypeError: non-default argument 'name' follows default argument 'id'


Solução moderna: `kw_only=True` na base (campos kw-only vão para o final do `__init__` e não sofrem a regra de ordem).

In [12]:
from datetime import datetime, timezone

@dataclass(kw_only=True)
class BaseEntity:
    id: int | None = None
    created_at: datetime = field(default_factory=lambda: datetime(2026, 11, 10, tzinfo=timezone.utc))

@dataclass
class Product(BaseEntity):
    name: str
    price: float

@dataclass
class DigitalProduct(Product):
    download_url: str

print(f'Product("Teclado", 300.0, id=1): {Product("Teclado", 300.0, id=1)}')
print(f'DigitalProduct("E-book", 49.9, "https://cdn/x.pdf"): {DigitalProduct("E-book", 49.9, "https://cdn/x.pdf")}')
import inspect
print(f"inspect.signature(DigitalProduct): {inspect.signature(DigitalProduct)}")

Product("Teclado", 300.0, id=1): Product(id=1, created_at=datetime.datetime(2026, 11, 10, 0, 0, tzinfo=datetime.timezone.utc), name='Teclado', price=300.0)
DigitalProduct("E-book", 49.9, "https://cdn/x.pdf"): DigitalProduct(id=None, created_at=datetime.datetime(2026, 11, 10, 0, 0, tzinfo=datetime.timezone.utc), name='E-book', price=49.9, download_url='https://cdn/x.pdf')
inspect.signature(DigitalProduct): (name: str, price: float, download_url: str, *, id: int | None = None, created_at: datetime.datetime = <factory>) -> None


## 7. `ClassVar`: atributos de classe

Anotações com `ClassVar` são **ignoradas** pelo dataclass: não viram campo, ficam compartilhadas pela classe.
Útil para constantes, contadores e registros.

In [13]:
@dataclass
class Ticket:
    PREFIX: ClassVar[str] = "TCK"
    _counter: ClassVar[int] = 0

    title: str
    code: str = field(init=False)

    def __post_init__(self):
        type(self)._counter += 1
        self.code = f"{self.PREFIX}-{self._counter:04d}"

print(f'Ticket("Erro no login"): {Ticket("Erro no login")}')
print(f'Ticket("Lentidão"): {Ticket("Lentidão")}')
print(f"campos: {[f.name for f in fields(Ticket)]}")

Ticket("Erro no login"): Ticket(title='Erro no login', code='TCK-0001')
Ticket("Lentidão"): Ticket(title='Lentidão', code='TCK-0002')
campos: ['title', 'code']


## 8. `fields()` e `metadata`

`fields()` devolve os objetos `Field` (nome, tipo, default, metadata...). O `metadata` é um dicionário **livre**:
o dataclass não o usa, mas o seu código pode. É assim que bibliotecas guardam "unidade", "coluna do banco",
"descrição para a UI", "regra de validação".

In [14]:
@dataclass
class SensorReading:
    sensor_id: str = field(metadata={"label": "Sensor"})
    temperature: float = field(metadata={"label": "Temperatura", "unit": "°C", "min": -40, "max": 85})
    humidity: float = field(metadata={"label": "Umidade", "unit": "%", "min": 0, "max": 100})

def render(obj) -> str:
    parts = []
    for f in fields(obj):
        unit = f.metadata.get("unit", "")
        parts.append(f"{f.metadata.get('label', f.name)}: {getattr(obj, f.name)}{unit}")
    return " | ".join(parts)

def check_ranges(obj) -> list[str]:
    errors = []
    for f in fields(obj):
        v, lo, hi = getattr(obj, f.name), f.metadata.get("min"), f.metadata.get("max")
        if lo is not None and not (lo <= v <= hi):
            errors.append(f"{f.name}={v} fora de [{lo}, {hi}]")
    return errors

r = SensorReading("s-01", 23.5, 104)
print(f"render(r): {render(r)}")
print(f"check_ranges(r): {check_ranges(r)}")

render(r): Sensor: s-01 | Temperatura: 23.5°C | Umidade: 104%
check_ranges(r): ['humidity=104 fora de [0, 100]']


O notebook avançado transforma essa ideia num validador genérico.

## 9. `match/case` com `__match_args__` (3.10+)

Dataclasses geram `__match_args__`, então funcionam com *pattern matching* posicional e por nome:

In [15]:
@dataclass(frozen=True)
class Circle:
    radius: float

@dataclass(frozen=True)
class Rect:
    w: float
    h: float

def describe(shape) -> str:
    match shape:
        case Circle(r) if r <= 0:
            return "círculo degenerado"
        case Circle(r):
            return f"círculo, área {3.14159 * r * r:.1f}"
        case Rect(w, h) if w == h:
            return f"quadrado {w}x{h}"
        case Rect(w=w, h=h):
            return f"retângulo {w}x{h}"
        case _:
            return "forma desconhecida"

print(f'[describe(s) for s in (Circle(2), Circle(0), Rect(3, 3), Rect(2, 5), "x")]: {[describe(s) for s in (Circle(2), Circle(0), Rect(3, 3), Rect(2, 5), "x")]}')

[describe(s) for s in (Circle(2), Circle(0), Rect(3, 3), Rect(2, 5), "x")]: ['círculo, área 12.6', 'círculo degenerado', 'quadrado 3x3', 'retângulo 2x5', 'forma desconhecida']


## 10. Configuração: TOML → dataclasses aninhadas

`tomllib` (stdlib, 3.11+) lê TOML para `dict`. A dataclass dá forma e imutabilidade.
O ponto que ninguém conta: **`AppConfig(**dict)` não converte as seções aninhadas** — elas continuam `dict`.

In [16]:
import tomllib

TOML = """
app_name = "inventory"
debug = false

[database]
host = "localhost"
port = 5432
pool_size = 10

[cache]
backend = "redis"
ttl_seconds = 300
"""

@dataclass(frozen=True)
class DatabaseConfig:
    host: str
    port: int
    pool_size: int = 5

@dataclass(frozen=True)
class CacheConfig:
    backend: str = "memory"
    ttl_seconds: int = 60

@dataclass(frozen=True)
class AppConfig:
    app_name: str
    database: DatabaseConfig
    cache: CacheConfig = field(default_factory=CacheConfig)
    debug: bool = False

raw = tomllib.loads(TOML)
naive = AppConfig(**raw)
print(f"type(naive.database): {type(naive.database)}")                # dict! naive.database.host quebraria

type(naive.database): <class 'dict'>


Precisamos de um loader explícito — e de validação manual. É aqui que fica claro por que `pydantic-settings` existe:

In [17]:
class ConfigError(ValueError):
    pass

def _require_type(section: str, key: str, value, expected: type):
    if not isinstance(value, expected) or (expected is int and isinstance(value, bool)):
        raise ConfigError(f"{section}.{key}: esperado {expected.__name__}, recebido {type(value).__name__} ({value!r})")

def load_config(data: dict) -> AppConfig:
    db = data.get("database") or {}
    for key, typ in (("host", str), ("port", int), ("pool_size", int)):
        if key in db:
            _require_type("database", key, db[key], typ)
    if not 1 <= db.get("port", 0) <= 65535:
        raise ConfigError("database.port fora do intervalo")
    cache = data.get("cache") or {}
    return AppConfig(
        app_name=data["app_name"],
        debug=bool(data.get("debug", False)),
        database=DatabaseConfig(**db),
        cache=CacheConfig(**cache),
    )

cfg = load_config(raw)
print(f"cfg: {cfg}")
print(f"cfg.database.host: {cfg.database.host}")
print(f"cfg.cache.ttl_seconds: {cfg.cache.ttl_seconds}")

try:
    load_config(tomllib.loads(TOML.replace("port = 5432", 'port = "5432"')))
except ConfigError as e:
    print(f"ConfigError: {e}")

cfg: AppConfig(app_name='inventory', database=DatabaseConfig(host='localhost', port=5432, pool_size=10), cache=CacheConfig(backend='redis', ttl_seconds=300), debug=False)
cfg.database.host: localhost
cfg.cache.ttl_seconds: 300
ConfigError: database.port: esperado int, recebido str ('5432')


Funciona, mas repare no custo: cada campo novo exige atualizar o loader, a checagem de tipo é manual, env vars e `.env`
não existem, e erros são reportados um por vez. **Regra prática:**

* configuração de **biblioteca/ferramenta** pequena → dataclass + `tomllib` + loader como acima;
* configuração de **aplicação** (serviço, API, job) → `pydantic-settings` (notebook `pydantic/02`).

## 11. JSON: serializar e desserializar

`asdict` + `json.dumps` cobre o caso simples. Tipos que o JSON não conhece (`datetime`, `Decimal`, `UUID`, `Enum`, `set`)
precisam de um `default=` customizado. A volta (JSON → dataclass) é manual.

In [18]:
import json
from decimal import Decimal
from enum import Enum
from uuid import UUID, uuid5, NAMESPACE_URL

class Status(Enum):
    OPEN = "open"
    PAID = "paid"

@dataclass
class Invoice:
    id: UUID
    total: Decimal
    status: Status
    issued_at: datetime
    tags: set[str] = field(default_factory=set)

def json_default(o):
    match o:
        case datetime():
            return o.isoformat()
        case Decimal():
            return str(o)                      # str preserva precisão; float não
        case UUID():
            return str(o)
        case Enum():
            return o.value
        case set() | frozenset():
            return sorted(o)
    raise TypeError(f"não serializável: {type(o).__name__}")

inv = Invoice(uuid5(NAMESPACE_URL, "inv-1"), Decimal("1234.50"), Status.PAID,
              datetime(2026, 11, 10, 9, 30, tzinfo=timezone.utc), {"b2b", "recorrente"})
payload = json.dumps(asdict(inv), default=json_default)
print(f"payload: {payload}")

payload: {"id": "2a7ecf2b-fdbd-5832-888f-655a26c51534", "total": "1234.50", "status": "paid", "issued_at": "2026-11-10T09:30:00+00:00", "tags": ["b2b", "recorrente"]}


In [19]:
def invoice_from_json(raw: str) -> Invoice:
    d = json.loads(raw)
    return Invoice(
        id=UUID(d["id"]),
        total=Decimal(d["total"]),
        status=Status(d["status"]),
        issued_at=datetime.fromisoformat(d["issued_at"]),
        tags=set(d.get("tags", [])),
    )

back = invoice_from_json(payload)
print(f"back == inv: {back == inv}")
print(f"back: {back}")

back == inv: True
back: Invoice(id=UUID('2a7ecf2b-fdbd-5832-888f-655a26c51534'), total=Decimal('1234.50'), status=<Status.PAID: 'paid'>, issued_at=datetime.datetime(2026, 11, 10, 9, 30, tzinfo=datetime.timezone.utc), tags={'b2b', 'recorrente'})


De novo: funciona, mas cada classe precisa do seu `from_json` escrito à mão. O notebook avançado constrói um
loader **genérico** usando os type hints — e compara com o Pydantic fazendo o mesmo em uma linha.

## Resumo

| recurso | quando usar |
|---|---|
| `frozen=True` | value objects, chaves de dict, configs, dados compartilhados entre threads |
| `tuple`/`frozenset` nos campos | imutabilidade profunda |
| `slots=True` | muitos objetos, atributos fixos, pegar typos |
| `kw_only` / `KW_ONLY` | construtores com muitos parâmetros; herança com defaults |
| `InitVar` | dado só para o construtor (senha, conexão) |
| `ClassVar` | constantes e contadores da classe |
| `fields()` + `metadata` | introspecção: UI, validação, mapeamento para banco |
| `match/case` | despacho por forma do objeto |
| config / JSON | possível com stdlib, mas manual — ponto de virada para Pydantic |

### Exercícios
1. Faça `Money` `frozen` + `order=True` e use como chave de dict.
2. Escreva `load_config` para um TOML com uma lista de `[[workers]]` (array of tables).
3. Adicione suporte a `date` (não só `datetime`) no `json_default` e escreva o teste de ida e volta.

➡️ Próximo: [`03_dataclass_avancado.ipynb`](03_dataclass_avancado.ipynb)